# Exercise 5.5 — Regularisation: lasso and ridge logistic regression (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day5/day5-05-regularisation.html>.*

*Open it in Jupyter, VS Code or Positron and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

> **A suggested addition, not part of the original module**
>
> This exercise has no page in the original version of the module. It covers the
> "Regularization" in the title of Day 5: the trees of Exercises 5.3 and 5.4 were
> restrained with hyperparameters, and this is the same idea for a regression
> model. Accept, rewrite or remove it; nothing else depends on it.

A logistic regression fitted without restraint chooses the coefficients that
fit the training rows best. **Regularisation** adds a price for large
coefficients, so that the model fits the training rows a little less closely in
exchange for being simpler. With predictors standardised, the fitted
coefficients minimise

$$\underbrace{-\frac{1}{n}\,\text{log-likelihood}}_{\text{misfit}} \;+\;
\lambda \times \underbrace{\textstyle\sum_j |\beta_j|}_{\text{lasso}}
\qquad\text{or}\qquad
\ldots \;+\; \lambda \times \underbrace{\tfrac{1}{2}\textstyle\sum_j \beta_j^2}_{\text{ridge}}$$

The **penalty** $\lambda$ is a hyperparameter: 0 gives the ordinary model, and
the larger it is, the more the coefficients are pulled towards zero. The
**lasso** pulls some of them all the way to zero, dropping those predictors;
**ridge** shrinks all of them but drops none.

**Theory:** penalised regression; lasso and ridge; why the predictors must be
standardised first; coefficient paths; choosing the penalty by
cross-validation.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score

The same people, split and five predictors as Exercises 5.2 to 5.4, from
[`data/diabetes.csv`](data/diabetes.csv). The pipelines standardise the
predictors: the penalty treats every coefficient alike, which is only fair if
the predictors are on the same scale.

In [ ]:
diabetes = pd.read_csv("data/diabetes.csv")
diabetes = diabetes[["pregnant", "glucose", "mass", "pedigree", "age", "diabetes"]]
diabetes = diabetes[(diabetes["glucose"] > 0) & (diabetes["mass"] > 0)]   # remove the impossible zeros
diabetes = diabetes.reset_index(drop=True)
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

predictors = ["pregnant", "glucose", "mass", "pedigree", "age"]

------------------------------------------------------------------------

### Part 1 — Coefficient paths

In scikit-learn a penalised logistic regression is a `LogisticRegression()`
with two arguments: `l1_ratio` chooses the kind of penalty, 1 for the lasso and
0 for ridge, and `C` sets its size the other way round: $C = 1 / (n\lambda)$,
so a **large** `C` means a **small** penalty. The `saga` solver handles both
kinds. Fit the lasso for 31 penalties from 0.001 to 1, one at a time:

In [ ]:
lambdas = 10 ** np.linspace(-3, 0, 31)
n = len(train)

rows = []
for lam in lambdas:
    lasso = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(l1_ratio=1, C=1 / (n * lam), solver="saga",
                                     max_iter=10000, random_state=123)),
    ])
    lasso = lasso.fit(train[predictors], train["diabetes"])
    for name, coef in zip(predictors, lasso["model"].coef_[0]):
        rows.append({"lambda": lam, "term": name, "estimate": coef})

lasso_path = pd.DataFrame(rows)
lasso_path[lasso_path["lambda"] == lambdas[20]].round(3)   # the coefficients at one penalty

Position 20 counted from the smallest penalty (Python counts from 0) is
$\lambda = 0.1$: only glucose has a coefficient that is not zero. The whole
path:

In [ ]:
fig, ax = plt.subplots()
for name in predictors:
    rows = lasso_path[lasso_path["term"] == name]
    ax.plot(rows["lambda"], rows["estimate"], linewidth=2, label=name)
ax.axhline(0, color="grey")
ax.set_xscale("log")
ax.set_xlabel("Penalty (log scale)")
ax.set_ylabel("Coefficient (per standard deviation)")
ax.set_title("Lasso: coefficients drop out one by one")
ax.legend()
plt.show()

Read the plot from right to left. At the largest penalties every coefficient is
zero. As the penalty falls, glucose enters first, then body mass, and the
others follow; at the smallest penalties the coefficients are close to those of
the ordinary logistic regression in Exercise 5.2. The order in which predictors
enter is one way of ranking them.

Now draw the same path for **ridge**:

> **Stuck?**
>
> `l1_ratio=1` is the lasso; ridge is the other end of the scale.

> **Worked answer**

In [ ]:
rows = []
for lam in lambdas:
    ridge = Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(l1_ratio=0, C=1 / (n * lam), solver="saga",
                                     max_iter=10000, random_state=123)),
    ])
    ridge = ridge.fit(train[predictors], train["diabetes"])
    for name, coef in zip(predictors, ridge["model"].coef_[0]):
        rows.append({"lambda": lam, "term": name, "estimate": coef})
ridge_path = pd.DataFrame(rows)

fig, ax = plt.subplots()
for name in predictors:
    path_rows = ridge_path[ridge_path["term"] == name]
    ax.plot(path_rows["lambda"], path_rows["estimate"], linewidth=2, label=name)
ax.axhline(0, color="grey")
ax.set_xscale("log")
ax.set_xlabel("Penalty (log scale)")
ax.set_ylabel("Coefficient (per standard deviation)")
ax.set_title("Ridge: every coefficient shrinks, none drops out")
ax.legend()
plt.show()

> No coefficient reaches zero. At the largest penalty, 1, glucose is still about
> 0.17 and the other four lie between about 0.04 and 0.09: ridge shrinks all the
> predictors together and keeps every one.

> **Learning points**
>
> - The penalty pulls coefficients towards zero; the larger it is, the simpler the model.
> - The lasso sets coefficients exactly to zero and so selects predictors; ridge shrinks them all and keeps every one.
> - Standardise the predictors first, or the penalty treats them unequally.

------------------------------------------------------------------------

### Part 2 — Choosing the penalty by cross-validation

The penalty is a hyperparameter, so it is chosen exactly like the tree depth in
Exercise 5.3: try a grid of values by 5-fold cross-validation on the training
rows, and keep the one with the best AUC. The grid is written in terms of `C`.

In [ ]:
lasso_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(l1_ratio=1, solver="saga", max_iter=10000, random_state=123)),
])

lasso_search = GridSearchCV(lasso_pipe, param_grid={"model__C": 1 / (n * lambdas)},
                            cv=5, scoring="roc_auc")
lasso_search = lasso_search.fit(train[predictors], train["diabetes"])

lasso_results = pd.DataFrame(lasso_search.cv_results_)
lasso_results["lambda"] = 1 / (n * lasso_results["param_model__C"].astype(float))
lasso_results = lasso_results[["lambda", "param_model__C", "mean_test_score", "std_test_score"]]
lasso_results.sort_values("mean_test_score", ascending=False).head(5).round(4)

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(lasso_results["lambda"], lasso_results["mean_test_score"],
            yerr=lasso_results["std_test_score"] / np.sqrt(5), marker="o", capsize=2)
ax.set_xscale("log")
ax.set_xlabel("Penalty (log scale)")
ax.set_ylabel("Mean cross-validated AUC")
ax.set_title("Cross-validated AUC along the lasso path")
plt.show()

The best penalty is 0.00126 in R (cross-validated AUC 0.805) and 0.0040 in
Python (`C` = 0.419, AUC 0.802); the folds differ between the languages, so the
choice differs too. The difference does not matter: the curve is flat at about
0.80 for every penalty up to about 0.02, falls slowly as predictors drop out,
and falls to 0.5, guessing, once the penalty (about 0.2 and above) sets every
coefficient to zero.

> **Learning points**
>
> - The penalty is tuned like any other hyperparameter: a grid, cross-validation on the training rows, the best AUC.
> - A flat curve means the data do not care much about the penalty: then prefer the simpler model it allows.

------------------------------------------------------------------------

### Part 3 — Does the penalty help on the test rows?

The search has refitted the lasso with the chosen penalty on all 600 training
rows. Fit the unpenalised model of Exercise 5.2 for comparison (`C=np.inf`):

In [ ]:
print(lasso_search.best_params_)
final_lasso = lasso_search.best_estimator_
print(pd.Series(final_lasso["model"].coef_[0], index=predictors).round(3))

plain = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(C=np.inf)),
])
plain = plain.fit(train[predictors], train["diabetes"])

Compute the test AUC of both:

> **Worked answer**

In [ ]:
lasso_prob = final_lasso.predict_proba(test[predictors])[:, 1]
plain_prob = plain.predict_proba(test[predictors])[:, 1]

print("lasso, test AUC:      ", round(roc_auc_score(test["diabetes"], lasso_prob), 3))
print("unpenalised, test AUC:", round(roc_auc_score(test["diabetes"], plain_prob), 3))

> R: 0.830 for both. Python: 0.829 for the lasso and 0.830 for the unpenalised
> model. At the chosen penalty the lasso has shrunk the coefficients only a
> little (glucose 1.01 in R and 0.989 in Python, against 1.02 unpenalised) and
> dropped none.

**Question!** With 600 people and five predictors the penalty buys little. In
what kind of data set would you expect it to matter much more?

> **Learning points**
>
> - With many people and few predictors, the unpenalised model is already stable and the penalty changes little.
> - Regularisation pays when there are many predictors for the number of people, as in omics data, where the lasso's variable selection is often the point.

## Stretch

> **Stretch — the noise test**
>
> Add five columns of pure random noise to the training and test rows
> (`rnorm()` in R, `np.random.default_rng(1).normal()` in Python), refit the
> unpenalised model and the tuned lasso on all ten predictors, and compare. Which
> model gives the noise columns coefficients, and which test AUC suffers more?